In [1]:
import pandas as pd
import numpy as np
import re
import os
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
career_df = pd.read_csv("../dataset/career_dataset_large_cleaned.csv")

career_df.head()

,Specialization,Certifications,CGPA/Percentage,Skills_ID_Categorized,Recommended_Career,Education_Level_ID_Categorized,Certifications_ID_Categorized
0,Finance,Tally ERP,67,"Konseling, MS Office, Machine Learning",Business Analyst,Sarjana,Keuangan & ERP
1,Science,AWS Certified,67,"Keuangan & Akuntansi, MS Office",Software Engineer,SMA/Sederajat,Cloud bersertifikat (AWS)
2,Business,Mental Health Basics,90,"Analisis Data & IT, Keuangan & Akuntansi, SQL",Financial Analyst,Magister,Kesehatan Mental
3,Computer Science,No Certification,75,Komunikasi,Staf Administrasi,Sarjana,Tanpa Sertifikat
4,Business,Tally ERP,83,Analisis Data & IT,Asisten Penjualan,SMA/Sederajat,Keuangan & ERP


In [10]:
career_df.shape

(4998, 7)

In [3]:
career_df = career_df.drop_duplicates().copy()

career_df.shape

(4998, 7)

In [12]:
print("Jumlah duplicate:", career_df.duplicated().sum())

Jumlah duplicate: 0


In [4]:
X = career_df["Skills_ID_Categorized"]
y = career_df["Recommended_Career"]

In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

X_cleaned = X.apply(clean_text)

X_cleaned.head()

0       konseling ms office machine learning
1               keuangan akuntansi ms office
2    analisis data it keuangan akuntansi sql
3                                 komunikasi
4                           analisis data it
Name: Skills_ID_Categorized, dtype: str

In [6]:
tfidf = TfidfVectorizer(max_features=1000)

X_tfidf = tfidf.fit_transform(X_cleaned)

X_tfidf.shape

(4998, 14)

In [13]:
print("Jumlah fitur TF-IDF:", X_tfidf.shape[1])

Jumlah fitur TF-IDF: 14


In [7]:
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print("Jumlah class:", len(encoder.classes_))
print(encoder.classes_)

Jumlah class: 12
['Akademisi/Dosen' 'Akuntan Junior' 'Asisten Penjualan' 'Business Analyst'
 'Eksekutif Pemasaran' 'Financial Analyst' 'Ilmuwan Peneliti'
 'Konselor Sekolah' 'ML Engineer' 'Operator Entri Data'
 'Software Engineer' 'Staf Administrasi']


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (3998, 14)
X_test : (1000, 14)
y_train: (3998,)
y_test : (1000,)


In [9]:
os.makedirs("../models", exist_ok=True)

joblib.dump(tfidf, "../models/tfidf_vectorizer_large.pkl")
joblib.dump(encoder, "../models/label_encoder_large.pkl")

['../models/label_encoder_large.pkl']

## Kesimpulan Preprocessing

Tahap preprocessing dan feature engineering berhasil dilakukan menggunakan kolom `Skills_ID_Categorized` sebagai fitur dan `Recommended_Career` sebagai target. Data teks dibersihkan, diubah menjadi representasi numerik menggunakan TF-IDF, dan label target dikonversi menggunakan LabelEncoder. Output tahap ini siap digunakan pada proses baseline modeling.